# Session 01: Introduction to Computer Vision

**CVI4IC - Summer Semester 2026**

In this notebook, we'll set up our environment and get a first taste of working with images using OpenCV and Python.

## 1. Environment Setup

First, let's install and import the required packages.

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Datasets

Throughout this course we use two main datasets:

1. **Fruits-360** — 174,700 images of 251 fruit/vegetable classes (100×100 px). Used for exercises in sessions 01–12.
2. **BAMBI** — Aerial wildlife monitoring imagery for the student project. Contains drone flights with annotations for roe deer, red deer, wild boar, and more.

### 2.1 Fruits-360

The [Fruits-360](https://github.com/fruits-360/) dataset contains 174,700 images of 251 classes (fruits, vegetables, nuts, seeds), all 100×100 px. It is organized into `Training/` and `Test/` splits by class name.

In [ ]:
# Clone the Fruits-360 dataset (shallow clone to save time)
!git clone --depth 1 https://github.com/fruits-360/fruits-360-100x100.git 2>/dev/null || echo "Dataset already cloned."

# Set dataset root
DATASET_ROOT = Path("fruits-360-100x100")
TRAIN_DIR = DATASET_ROOT / "Training"
TEST_DIR = DATASET_ROOT / "Test"

# List available classes
classes = sorted(os.listdir(TRAIN_DIR))
print(f"\nFruits-360: {len(classes)} classes found")
print(f"First 10 classes: {classes[:10]}")

### 2.2 BAMBI — Aerial Wildlife Monitoring (Project Dataset)

The [BAMBI](https://github.com/bambi-eco/Dataset) dataset contains drone imagery for wildlife monitoring. You will use this dataset for your **course project**.

**Download steps:**
1. Clone the dataset repository (contains metadata and download scripts)
2. Filter flights by species of interest (e.g., roe deer, red deer, wild boar)
3. Download the selected flights from Zenodo using the provided script

> The `-f` flag specifies which flight IDs to download. Adjust the species filter and flight IDs to match your project needs.

In [ ]:
# Step 1: Clone the BAMBI dataset repository
!git clone https://github.com/bambi-eco/Dataset.git 2>/dev/null || echo "BAMBI repo already cloned."

# Step 2: Filter flights by species of interest
!python Dataset/filter_flights.py --folder Dataset/flight_metadata/ --species "Roe deer" "Red deer" "Wild boar"

# Step 3: Download selected flights from Zenodo
# -f specifies flight IDs to download (e.g., 0 1 2)
!python Dataset/download_from_zenodo.py \
    -s Dataset/flight_metadata/zenodo_upload_summary.json \
    -f 0 1 \
    -t cUsHQu9a2ABEda5ZvOFcpiQfmfAmRT1q2ug3K2JD3La4b6DucfMiKOCzc1sV

### 2.3 BAMBI — ALFS Data

The ALFS (Airborne Light Field Sampling) subset of BAMBI contains pre-cropped detection images. This is a single download — no flight filtering needed.

In [ ]:
# TODO: Download link will be provided by your instructor
# !wget -q <ALFS_DOWNLOAD_URL> -O alfs_data.zip
# !unzip -q alfs_data.zip -d alfs_data/

## 3. Loading and Displaying Images

Let's load some fruit images from the dataset and explore their properties.

In [ ]:
# Load a fruit image
apple_dir = TRAIN_DIR / "Apple Red 1"
img_path = sorted(apple_dir.glob("*.jpg"))[0]
img = cv2.imread(str(img_path))

print(f"Image path: {img_path.name}")
print(f"Image shape: {img.shape}  (height, width, channels)")
print(f"Image dtype: {img.dtype}")
print(f"Pixel value range: [{img.min()}, {img.max()}]")

In [ ]:
# Display with matplotlib (OpenCV loads as BGR — convert to RGB)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(4, 4))
plt.imshow(img_rgb)
plt.title(f"Apple Red 1 — {img.shape[1]}×{img.shape[0]} px")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Display a grid of different fruit classes
sample_classes = ["Apple Red 1", "Banana 1", "Cherry 1", "Lemon 1", "Orange 1", "Strawberry 1", "Tomato 1", "Pear 1"]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for ax, cls_name in zip(axes.flat, sample_classes):
    cls_dir = TRAIN_DIR / cls_name
    img_path = sorted(cls_dir.glob("*.jpg"))[0]
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(cls_name, fontsize=10)
    ax.axis("off")

plt.suptitle("Fruits-360 — Sample Classes", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Basic Image Operations

In [ ]:
# Access individual pixel values (center of the 100x100 image)
pixel = img[50, 50]
print(f"Pixel at (50, 50): B={pixel[0]}, G={pixel[1]}, R={pixel[2]}")

# Crop a region (the fruit is centered in the image)
crop = img[20:80, 20:80]
print(f"Cropped region shape: {crop.shape}")

# Resize to a larger image
resized = cv2.resize(img, (256, 256))
print(f"Resized shape: {resized.shape}")

# Show original, cropped, and resized side by side
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original ({img.shape[1]}×{img.shape[0]})")
axes[1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Cropped ({crop.shape[1]}×{crop.shape[0]})")
axes[2].imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Resized ({resized.shape[1]}×{resized.shape[0]})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Drawing on Images

In [ ]:
# Draw bounding boxes and labels on a fruit image
img_draw = cv2.resize(img.copy(), (256, 256))

# Simulate a detection bounding box around the fruit
cv2.rectangle(img_draw, (40, 40), (216, 216), (0, 255, 0), 2)
cv2.putText(img_draw, "Apple", (40, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

# Draw a circle at the center
cv2.circle(img_draw, (128, 128), 5, (0, 0, 255), -1)

plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(img_draw, cv2.COLOR_BGR2RGB))
plt.title("Drawing on Images: Bounding Box & Label")
plt.axis("off")
plt.tight_layout()
plt.show()

## Exercises

1. **Explore the dataset:** Pick 3 fruit classes of your choice, load one image from each, and display them side by side. Print the number of images per class.
2. **Grid overlay:** Write a function `draw_grid(img, step=10)` that draws a grid overlay on any image. Test it on a fruit image.
3. **Resize comparison:** Load a fruit image and resize it to 32×32 using `cv2.INTER_NEAREST`, `cv2.INTER_LINEAR`, and `cv2.INTER_CUBIC`. Display all three results scaled back up to 256×256 — can you see the difference?

In [ ]:
# Your code here
